# AI Code Review: Отчёт по домашнему заданию

**Тема:** ИИ как первый ревьюер — фильтрация замечаний, классификация по схеме «светофор», оформление PR-комментариев.

**Преподаватель:** Владимир Языков  
**Языки:** Python, Java, Go  
**Формат:** Итоговый отчёт с промптами, ответами ИИ, классификацией, комментариями и next steps.

---
## Подготовка окружения

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

def show_table(df):
    """Отобразить DataFrame с подсветкой приоритетов."""
    def color_priority(val):
        colors = {'blocker': 'background-color: #f8d7da',
                  'major': 'background-color: #fff3cd',
                  'minor': 'background-color: #d6d8d9'}
        return colors.get(val, '')
    
    def color_light(val):
        colors = {'Accept': 'background-color: #d4edda',
                  'Verify': 'background-color: #fff3cd',
                  'Reject': 'background-color: #f8d7da'}
        return colors.get(val, '')
    
    return df.style.map(color_priority, subset=['Приоритет']) \
                   .map(color_light, subset=['Светофор'])

print("[x] Окружение готово. pandas", pd.__version__)

---

## Пример 1 — Python: Command Injection

**Ситуация:** Утилита запускает системную команду. Аргумент `folder` приходит от пользователя (из формы).

In [ ]:
# Исходный код (Python)
PYTHON_CODE = '''import os

def run_backup(folder: str) -> int:
    cmd = f"tar -czf backup.tgz {folder}"
    return os.system(cmd)'''

print(PYTHON_CODE)

### Промпт

In [ ]:
PYTHON_PROMPT = '''You are a senior Python security engineer reviewing a DevOps backup script.

Context:
- The function `run_backup` takes a folder path from user input and constructs a shell command using an f-string.
- The script runs on a Linux server.

Criteria (check in this order):
1. Security: command injection, path traversal, unsafe subprocess calls.
2. Correctness: does the function behave correctly for edge cases (spaces, special chars)?
3. Maintainability: use of standard library, error handling, typing.
4. Testability: is the function easy to unit test?

Constraints:
- Do NOT suggest switching to a different programming language.
- Do NOT suggest major refactoring without justification.
- Do NOT invent non-existent libraries or APIs.

Output format — for each finding use EXACTLY this structure:
[BLOCKER | MAJOR | MINOR] <short description>\nRisk: <why it matters>\nFix: <concrete code suggestion>\n'''

print(PYTHON_PROMPT)

### Ответ ИИ (симулированный)

In [ ]:
PYTHON_AI_RESPONSE = '''\
[BLOCKER] Command injection via unsanitized user input in f-string
Risk: An attacker can inject arbitrary shell commands by passing "; rm -rf /" as the folder name.
Fix: Use subprocess.run() with a list argument (no shell=True) and validate the input.

[MAJOR] Deprecated os.system() used instead of subprocess
Risk: os.system() invokes a shell, which increases attack surface and is less controllable.
Fix: Replace with subprocess.run(['tar', '-czf', 'backup.tgz', folder], check=True).

[MAJOR] Missing error handling: os.system returns exit code but it's not checked
Risk: Caller receives 0 even if tar fails silently; no exception is raised on failure.
Fix: Use subprocess.run(... check=True) which raises CalledProcessError on non-zero exit.

[MAJOR] Input validation is missing for the folder parameter
Risk: Even without injection, spaces or shell metacharacters break the command.
Fix: Validate folder is an existing directory via os.path.isdir() before proceeding.

[MINOR] Type hint suggests folder is used as a path but no pathlib
Risk: Low; but using pathlib.Path would make path manipulation safer and cleaner.
Fix: Consider using pathlib.Path(folder) and subprocess for a modern approach.

[MINOR] Function returns int but callers may expect None or bool
Risk: Minor API inconsistency; return type is not semantic.
Fix: Consider returning bool (True on success) or raising exceptions.
'''

print(PYTHON_AI_RESPONSE)

### Классификация замечаний

In [ ]:
py_classification = pd.DataFrame([
    {
        'Замечание ИИ': 'Command injection via unsanitized user input in f-string',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'Немедленно исправить: заменить os.system на subprocess.run с list-аргументом. Добавить валидацию входа.'
    },
    {
        'Замечание ИИ': 'Deprecated os.system() used instead of subprocess',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Принять и исправить вместе с первым пунктом (замена на subprocess.run).'
    },
    {
        'Замечание ИИ': 'Missing error handling: os.system returns exit code but it\'s not checked',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Добавить check=True в subprocess.run() или явную проверку returncode.'
    },
    {
        'Замечание ИИ': 'Input validation is missing for the folder parameter',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Добавить os.path.isdir() проверку. Если папка не существует — вернуть ошибку с сообщением.'
    },
    {
        'Замечание ИИ': 'Type hint suggests folder is used as a path but no pathlib',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'pathlib — хорошая практика, но в короткой утилите избыточно. Проверить код-стайл проекта. Если проект использует pathlib — принять.'
    },
    {
        'Замечание ИИ': 'Function returns int but callers may expect None or bool',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'Проверить контракты текущих вызывающих кодов. Если возврат int ожидается — оставить. Иначе изменить сигнатуру.'
    },
])

show_table(py_classification)

### Итоговые комментарии (PR-style)

In [ ]:
PYTHON_PR_COMMENTS = '''\
**[BLOCKER] Command injection через пользовательский ввод**
Переменная `folder` подставляется в shell-команду через f-string без экранирования. Злоумышленник может передать `; rm -rf /`.
Риск: Полная компрометация сервера.
Исправить: `subprocess.run(['tar', '-czf', 'backup.tgz', folder], check=True)` — shell не участвует, injection невозможен.

**[MAJOR] Отсутствует валидация входного параметра**
Функция не проверяет, существует ли folder и является ли он директорией.
Риск: Ошибки выполнения, неинформативные сообщения пользователю.
Исправить: Добавить `if not os.path.isdir(folder): raise ValueError(f\"Directory {folder} not found\")`.

**[MAJOR] Нет обработки ошибок при выполнении команды**
`os.system()` возвращает exit code, но он не проверяется. Успешность вызова не гарантирована.
Риск: Функция возвращает 0 даже при неудачном backup.
Исправить: Использовать `subprocess.run(..., check=True)` — при ошибке будет выброшено исключение.
'''

display(Markdown(PYTHON_PR_COMMENTS))

### Next steps

In [ ]:
PYTHON_NEXT_STEPS = '''\
**До merge:**
- Заменить os.system -> subprocess.run с list-аргументом.
- Добавить валидацию folder через os.path.isdir().
- Проверить обработку ошибок (check=True или явная проверка returncode).

**В отдельную задачу:**
- Рассмотреть миграцию на pathlib для единого стиля работы с путями.
- Уточнить возвращаемый тип (int vs bool) — проверить вызывающие коды.

**Тесты:**
- Добавить unit-тест с валидным путём.
- Добавить unit-тест с несуществующим путём (ожидать исключение).
- Добавить unit-тест с путём, содержащим спецсимволы (пробелы, $, `).
'''

display(Markdown(PYTHON_NEXT_STEPS))

---

## Пример 2 — Java: NullPointerException

**Ситуация:** Сервис отправляет welcome-письмо. Репозиторий может вернуть `null`, если пользователь не найден.

In [ ]:
JAVA_CODE = '''public class MailService {
    private final UserRepository userRepository;
    private final MailClient mailClient;

    public MailService(UserRepository userRepository, MailClient mailClient) {
        this.userRepository = userRepository;
        this.mailClient = mailClient;
    }

    public void sendWelcomeEmail(Long userId) {
        User user = userRepository.findById(userId);
        mailClient.send(user.getEmail(), "Welcome!");
    }
}'''

print(JAVA_CODE)

### Промпт

In [ ]:
JAVA_PROMPT = '''You are a senior Java developer doing a code review for an email service module.

Context:
- `MailService.sendWelcomeEmail` finds a User by ID and sends a welcome email.
- `userRepository.findById()` may return null according to its contract.

Criteria:
1. Null safety: handle null returns, avoid NullPointerException.
2. Exception handling: what happens if sending fails?
3. Code quality: naming, constructor injection, SOLID principles.
4. Testability: is the class easy to mock and test?

Constraints:
- Do NOT suggest switching to a different language or framework.
- Do NOT suggest Lombok, Project Reactor, or external libraries unless critically needed.
- Output format: [BLOCKER | MAJOR | MINOR] <description>\\nRisk: <risk>\\nFix: <fix>\n'''

print(JAVA_PROMPT)

### Ответ ИИ (симулированный)

In [ ]:
JAVA_AI_RESPONSE = '''\
[BLOCKER] NullPointerException when userRepository.findById() returns null
Risk: Calling .getEmail() on null crashes the service with NPE; no error logged.
Fix: Add null check: if (user == null) { throw new UserNotFoundException(userId); } or use Optional<User>.

[MAJOR] User entity exposes getEmail() without null safety on email field
Risk: If User has no email, getEmail() might return null, causing NPE in mailClient.send(null, ...).
Fix: Ensure User.email is non-null (validation in constructor) or add null check before sending.

[MAJOR] Missing exception handling for mailClient.send()
Risk: If SMTP server is down, the exception propagates unhandled, causing 500 error to client.
Fix: Wrap in try-catch, log the error, and return a meaningful error response.

[MINOR] Class fields should be final — they already are. Good.
Risk: None. This is a positive observation, not an issue.
Fix: None needed.

[MAJOR] Use @Override on constructor injection
Risk: No risk; constructor is not overriding anything. Adding @Override here would be incorrect.
Fix: Do NOT add @Override to constructors in Java — it is a syntax error.

[MINOR] sendWelcomeEmail could be renamed to sendWelcomeEmailToUser for clarity
Risk: Low; naming preference.
Fix: Rename if team convention prefers more explicit names.
'''

print(JAVA_AI_RESPONSE)

### Классификация замечаний

In [ ]:
java_classification = pd.DataFrame([
    {
        'Замечание ИИ': 'NullPointerException when userRepository.findById() returns null',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'Добавить null-check перед вызовом user.getEmail(). Выбросить кастомное исключение UserNotFoundException.'
    },
    {
        'Замечание ИИ': 'User entity exposes getEmail() without null safety on email field',
        'Тип': 'корректное',
        'Светофор': 'Verify',
        'Приоритет': 'major',
        'Что делаем': 'Проверить конструктор User — гарантируется ли там non-null email? Если да — Reject (ложное срабатывание). Если нет — Accept и добавить валидацию.'
    },
    {
        'Замечание ИИ': 'Missing exception handling for mailClient.send()',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Обернуть send() в try-catch, залогировать ошибку, не дать упасть всему запросу.'
    },
    {
        'Замечание ИИ': 'Class fields should be final — they already are. Good.',
        'Тип': 'ошибочное',
        'Светофор': 'Reject',
        'Приоритет': 'minor',
        'Что делаем': 'Это не замечание, а похвала. Не требует действий. В ревью-комментарии не включаем.'
    },
    {
        'Замечание ИИ': 'Use @Override on constructor injection',
        'Тип': 'ошибочное',
        'Светофор': 'Reject',
        'Приоритет': 'major',
        'Что делаем': 'В Java @Override над конструктором — синтаксическая ошибка. ИИ дал неверный совет. Отклонить.'
    },
    {
        'Замечание ИИ': 'sendWelcomeEmail could be renamed to sendWelcomeEmailToUser for clarity',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'Проверить код-стайл проекта. Если в проекте краткие имена в сервисах — оставить. Иначе переименовать.'
    },
])

show_table(java_classification)

### Итоговые комментарии (PR-style)

In [ ]:
JAVA_PR_COMMENTS = '''\
**[BLOCKER] NPE при отсутствии пользователя в БД**
`userRepository.findById(userId)` может вернуть null, но код вызывает `user.getEmail()` без проверки.
Риск: Crash сервиса при каждом обращении с невалидным userId. Отсутствует логирование.
Исправить: `if (user == null) { log.warn(\"User {} not found\", userId); return; }` или выбросить `UserNotFoundException`.

**[MAJOR] Нет обработки ошибок mailClient.send()**
Если SMTP недоступен, исключение пробрасывается наружу без логирования.
Риск: Потеря ошибок, невозможность диагностировать проблемы с почтой.
Исправить: Обернуть в try-catch, логировать, вернуть пользователю friendly error.

**[MAJOR] Возможен null в email-поле User**
Если конструктор User допускает null email, то mailClient.send(null, ...) упадёт.
Риск: NPE с ложной трассировкой (будут искать проблему в MailService, а не в User).
Исправить: Добавить requireNonNull(email) в конструктор User или добавить null-check перед send().
'''

display(Markdown(JAVA_PR_COMMENTS))

### Next steps

In [ ]:
JAVA_NEXT_STEPS = '''\
**До merge:**
- Добавить null-check после findById().
- Добавить try-catch вокруг mailClient.send().

**В отдельную задачу:**
- Проверить конструктор User на nullable-поля. Если email может быть null — добавить валидацию на уровне entity.
- Обсудить нейминг на код-ревью (sendWelcomeEmail vs sendWelcomeEmailToUser).

**Тесты:**
- Юнит-тест: findById возвращает null -> ожидать UserNotFoundException.
- Юнит-тест: mailClient.send кидает исключение -> проверить логирование.
- Интеграционный тест: вызов sendWelcomeEmail с валидным userId -> письмо отправлено.
'''

display(Markdown(JAVA_NEXT_STEPS))

---

## Пример 3 — Go: SQL-инъекция

**Ситуация:** Обработчик HTTP-запроса принимает параметр `status` и конкатенирует его в SQL-запрос.

In [ ]:
GO_CODE = '''package handlers

import (
    "database/sql"
    "fmt"
    "net/http"
)

func Orders(db *sql.DB, w http.ResponseWriter, r *http.Request) error {
    status := r.URL.Query().Get("status")
    query := fmt.Sprintf("SELECT id, total FROM orders WHERE status = '%s'", status)
    _, err := db.Query(query)
    return err
}'''

print(GO_CODE)

### Промпт

In [ ]:
GO_PROMPT = '''You are a senior Go backend developer reviewing an order processing handler.

Context:
- HTTP handler `Orders` reads a `status` query parameter and constructs an SQL query by string concatenation (fmt.Sprintf).
- Uses a hypothetical `database/sql` library.

Criteria:
1. Security: SQL injection, input validation, parameterized queries.
2. Correctness: HTTP handler signature, error handling, response codes.
3. Code quality: follows Go idioms, proper use of standard library.
4. Error handling: what does returning err do to the HTTP response?

Constraints:
- Do NOT suggest switching to an ORM or different language.
- Assume standard Go conventions.
- Only suggest library changes if they are standard and justified.
- Output format: [BLOCKER | MAJOR | MINOR] <description>\\nRisk: <risk>\\nFix: <fix>\n'''

print(GO_PROMPT)

### Ответ ИИ (симулированный)

In [ ]:
GO_AI_RESPONSE = '''\
[BLOCKER] SQL injection via string formatting of user input
Risk: Attacker can inject SQL by passing status=1' OR '1'='1, dumping all orders.
Fix: Use parameterized queries: db.Query("SELECT id, total FROM orders WHERE status = ?", status).

[MAJOR] Error handling returns raw database error to HTTP client
Risk: Exposing internal SQL errors (table names, column names) to users is a security issue.
Fix: Return a generic 500 error and log the actual error server-side.

[MAJOR] No HTTP status code set on success
Risk: Client receives no explicit 200 OK; some clients may treat empty response as error.
Fix: Call w.WriteHeader(http.StatusOK) or equivalent before returning.

[MAJOR] Input validation missing for status parameter
Risk: Even with parameterized queries, valid status values should be constrained to known enum values.
Fix: Validate status against a whitelist: validStatuses = ["pending", "shipped", "delivered"] before querying.

[MINOR] Use strings.ToLower() for case-insensitive matching
Risk: If user passes "PENDING", query returns no results due to case mismatch.
Fix: Normalize input with strings.ToLower(status) before validation or query.

[MINOR] Consider using camelCase for the function name — Go uses camelCase by convention
Risk: None; this is factually incorrect. Go standard library uses camelCase for functions.
Fix: Actually Go standard library uses camelCase (e.g., strings.HasPrefix). The suggestion is correct but stated confusingly.
'''

print(GO_AI_RESPONSE)

### Классификация замечаний

In [ ]:
go_classification = pd.DataFrame([
    {
        'Замечание ИИ': 'SQL injection via string formatting of user input',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'Переписать на parameterized query: заменить fmt.Sprintf на db.Query() с плейсхолдером ? и передачей параметра отдельно.'
    },
    {
        'Замечание ИИ': 'Error handling returns raw database error to HTTP client',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Заменить return err на логирование и возврат generic 500 ошибки.'
    },
    {
        'Замечание ИИ': 'No HTTP status code set on success',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Добавить w.WriteHeader(200) перед возвратом данных.'
    },
    {
        'Замечание ИИ': 'Input validation missing for status parameter',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'Добавить whitelist-валидацию: известные значения status. При несовпадении — 400 Bad Request.'
    },
    {
        'Замечание ИИ': 'Use strings.ToLower() for case-insensitive matching',
        'Тип': 'корректное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'Проверить, ожидаются ли значения в разном регистре. Если да — добавить нормализацию.'
    },
    {
        'Замечание ИИ': 'Consider using camelCase for the function name',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'Проверить конвенцию проекта. Go stdlib использует camelCase (strings.HasPrefix), но многие Go-проекты используют snake_case для функций. Решение: следовать проектной конвенции.'
    },
])

show_table(go_classification)

### Итоговые комментарии (PR-style)

In [ ]:
GO_PR_COMMENTS = '''\
**[BLOCKER] SQL-инъекция через fmt.Sprintf**
Параметр `status` напрямую подставляется в SQL-запрос через форматирование строки.
Риск: Злоумышленник может передать `1' OR '1'='1` и получить все заказы.
Исправить: Использовать parameterized query: `db.Query("SELECT id, total FROM orders WHERE status = ?", status)`.

**[MAJOR] Утечка внутренней ошибки БД клиенту**
Функция возвращает `err` напрямую в HTTP-ответ. Это раскрывает структуру БД.
Риск: Information disclosure. Злоумышленник видит имена таблиц и колонок.
Исправить: Залогировать ошибку, вернуть generic 500.

**[MAJOR] Отсутствует HTTP-статус при успехе**
Нет вызова w.WriteHeader(200), клиент не получает явного OK.
Риск: Некорректная работа клиентов, ожидающих статус-код.
Исправить: Добавить `w.WriteHeader(http.StatusOK)` перед завершением.

**[MAJOR] Нет валидации параметра status**
Любое строковое значение принимается и идёт в запрос.
Риск: Лишние обращения к БД, потенциально некорректные данные.
Исправить: Валидировать по белому списку: `["pending", "shipped", "delivered"]`.
'''

display(Markdown(GO_PR_COMMENTS))

### Next steps

In [ ]:
GO_NEXT_STEPS = '''\
**До merge:**
- Переписать запрос на parameterized query (существенно — blocker).
- Добавить валидацию status по whitelist + 400 Bad Request при несовпадении.
- Убрать return err, добавить логирование + generic 500.
- Добавить w.WriteHeader(200) при успехе.

**В отдельную задачу:**
- Уточнить нейминг-конвенцию (camelCase vs snake_case для функций в проекте).
- Решить, нужна ли нормализация регистра для status (strings.ToLower).

**Тесты:**
- Юнит-тест: валидный status -> запрос выполнен, 200 OK.
- Юнит-тест: невалидный status -> 400 Bad Request.
- Юнит-тест: SQL-инъекция в status -> parameterized query не даёт injection.
- Интеграционный тест: ошибка БД -> 500 + лог.
'''

display(Markdown(GO_NEXT_STEPS))

---

## Сводная таблица классификации

In [ ]:
all_data = pd.concat([
    py_classification.assign(Пример='1. Python'),
    java_classification.assign(Пример='2. Java'),
    go_classification.assign(Пример='3. Go'),
])

summary = all_data.groupby(['Пример', 'Светофор']).agg(
    Количество=('Замечание ИИ', 'count'),
    Приоритеты=('Приоритет', lambda x: ', '.join(sorted(x)))
).reset_index()

display(Markdown("### Распределение по светофору"))
display(summary)

display(Markdown("### Полная таблица"))
show_table(all_data[['Пример', 'Замечание ИИ', 'Тип', 'Светофор', 'Приоритет']])

### Чек-лист самопроверки

In [ ]:
checklist = pd.DataFrame([
    ['[x]', 'Для каждого из 3 примеров есть промпт, ответ ИИ и классификация'],
    ['[x]', 'В промптах указан контекст, критерии качества, формат ответа и приоритеты'],
    ['[x]', 'Для каждого замечания есть тип (корректное/сомнительное/ошибочное) и светофор (Accept/Verify/Reject)'],
    ['[x]', 'Для Verify указано, как проверить'],
    ['[x]', 'Итоговые комментарии: что не так -> риск -> как исправить -> приоритет'],
    ['[x]', 'Для каждого примера есть next steps: до merge, отдельная задача, тесты'],
], columns=['Статус', 'Критерий'])

display(checklist)

---

**Вывод:** Все 3 примера обработаны. Каждый содержит промпт, ответ ИИ, классификацию замечаний, PR-комментарии и next steps.  
Приоритеты и светофор обоснованы. ИИ-замечания не приняты на веру — для Verify указан способ проверки, Reject аргументирован.